# DJIA Historical Dataset Builder — v11
## Dataset completo con metriche fondamentali mensili per ogni titolo

**Aggiornamenti v11:**
- Metriche fondamentali trimestrali scaricate da yfinance e riproiettate mensilmente (point-in-time)
- P/E, P/B, EV/EBITDA, ROE, ROA, Debt/Equity, Margini — tutti calcolati mese per mese
- Copertura diagnostica dettagliata: quanti titoli hanno dati e per quale periodo
- RF da FRED (TB3MS) — nessun valore hardcoded
- Solo Parte 1 (dataset builder)


---
## 1. Installazione Librerie


## 1. Installazione delle librerie

Questa cella installa le dipendenze minime usate nel builder.  
Se lavori in locale e hai già tutto installato, puoi anche saltarla.

In [1]:
!pip install -q yfinance pandas numpy openpyxl fredapi

---
## 2. Import e Configurazione

Tutti i parametri globali sono definiti qui in un unico posto per facilitare le modifiche.


## 2. Import, warning e configurazione iniziale

Qui importiamo le librerie principali e definiamo i parametri globali del notebook.  
Questa è la sezione di setup generale dell'ambiente.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
from io import StringIO

# ── Parametri globali ─────────────────────────────────────────────────────────
START_DATE = "2005-01-01"
END_DATE   = "2026-03-30"

# RF rate: caricato da FRED nella cella successiva (TB3MS)
# NON usare valori hardcoded — tutto viene da FRED

print("✓ Configurazione caricata.")
print(f"  Periodo: {START_DATE}  →  {END_DATE}")
print("  RF rate: verrà scaricato da FRED (cella successiva)")


✓ Configurazione caricata.
  Periodo: 2005-01-01  →  2026-04-30
  Anni coperti dal risk-free: 2005 – 2026


## 2b. FRED API — Risk-Free Rate e Macro Features

Scarichiamo i dati ufficiali dalla Federal Reserve (FRED):
- **TB3MS**: T-Bill 3 mesi → Risk-Free Rate mensile reale
- **DGS10 / DGS2**: yield curve (slope = 10y − 2y)
- **BAMLH0A0HYM2**: HY credit spread (proxy risk appetite)
- **CPIAUCSL**: inflazione YoY
- **UNRATE**: tasso di disoccupazione (variazione mensile)
- **USREC**: dummy recessione NBER
- **^VIX** (yfinance): volatilità implicita mercato
- **DX-Y.NYB** (yfinance): Dollar Index


In [ ]:
from fredapi import Fred

FRED_API_KEY = "7fffa6b224119d62c4bf797d75b92e33"
fred = Fred(api_key=FRED_API_KEY)

def fred_monthly(series_id, start=START_DATE, end=END_DATE, transform=None):
    """Scarica serie FRED, resample mensile, forward-fill."""
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s = s.resample('ME').last().ffill()
    s.index = s.index.to_period('M').to_timestamp('M')
    if transform == 'yoy':
        s = s.pct_change(12)
    elif transform == 'diff':
        s = s.diff()
    return s

print("Downloading FRED series...")

# ── 1) Risk-free rate mensile reale ────────────────────────────────────────────
# TB3MS = T-Bill 3 mesi, annualizzato → dividiamo per 12 per ottenere il tasso
# mensile come decimale (es: 5.1% annuo → 0.051/12 = 0.00425 mensile)
tb3ms = fred_monthly('TB3MS')
rf_monthly_fred = tb3ms / 100 / 12          # mensile, decimale
print(f"  TB3MS: {tb3ms.index[0].date()} – {tb3ms.index[-1].date()} ({len(tb3ms)} obs)")
print(f"  Esempio RF mensile: {rf_monthly_fred.iloc[-1]:.5f} (= {tb3ms.iloc[-1]:.2f}% annuo / 12)")

# ── 2) Yield curve: slope 10y − 2y ─────────────────────────────────────────────
dgs10 = fred_monthly('DGS10')
dgs2  = fred_monthly('DGS2')
yield_curve = (dgs10 - dgs2).rename('YieldCurve_10y2y')

# ── 3) HY credit spread ────────────────────────────────────────────────────────
hy_spread = fred_monthly('BAMLH0A0HYM2').rename('HY_Spread')

# ── 4) Inflazione YoY ──────────────────────────────────────────────────────────
cpi_yoy = fred_monthly('CPIAUCSL', transform='yoy').rename('CPI_YoY')

# ── 5) Variazione tasso disoccupazione ─────────────────────────────────────────
unrate_chg = fred_monthly('UNRATE', transform='diff').rename('Unrate_Chg')

# ── 6) Dummy recessione NBER ───────────────────────────────────────────────────
recession = fred_monthly('USREC').rename('Recession')

# ── 7) DJIA reale da FRED (benchmark di confronto) ─────────────────────────────
# Serie DJIA su FRED: giornaliera, prendiamo l'ultimo valore del mese
djia_fred_raw = fred.get_series('DJIA', observation_start=START_DATE, observation_end=END_DATE)
djia_fred = djia_fred_raw.resample('ME').last().ffill()
djia_fred.index = djia_fred.index.to_period('M').to_timestamp('M')
djia_fred.name = 'DJIA_Real'
print(f"  DJIA reale (FRED): {djia_fred.index[0].date()} – {djia_fred.index[-1].date()}")
print(f"  Ultimo valore: {djia_fred.iloc[-1]:,.0f}")

# ── Assembla macro_df ──────────────────────────────────────────────────────────
idx = rf_monthly_fred.index
macro_df = pd.DataFrame(index=idx)
macro_df['RF_Monthly_FRED']  = rf_monthly_fred
macro_df['YieldCurve_10y2y'] = yield_curve.reindex(idx)
macro_df['HY_Spread']        = hy_spread.reindex(idx)
macro_df['CPI_YoY']          = cpi_yoy.reindex(idx)
macro_df['Unrate_Chg']       = unrate_chg.reindex(idx)
macro_df['Recession']        = recession.reindex(idx)
macro_df['DJIA_Real']        = djia_fred.reindex(idx)

print(f"\n✓ macro_df: {macro_df.shape}")

# ── 8) DJIA reale esteso: Excel 1928-2014 + FRED 2014+ ─────────────────────
# Il file PriceHistory-2.xlsx ha dati DJIA giornalieri 1928-2014.
# Lo usiamo come ground truth per il periodo 2005-2014, poi suturiamo con FRED.
try:
    _xls = pd.read_excel('PriceHistory-2.xlsx', sheet_name=0)
    # Normalizza colonne
    _xls.columns = [str(c).strip() for c in _xls.columns]
    date_col  = [c for c in _xls.columns if 'date' in c.lower() or 'data' in c.lower()][0]
    price_col = [c for c in _xls.columns if c != date_col][0]
    _xls[date_col] = pd.to_datetime(_xls[date_col], dayfirst=True, errors='coerce')
    _xls = _xls.dropna(subset=[date_col]).sort_values(date_col)
    _xls[price_col] = pd.to_numeric(_xls[price_col], errors='coerce')
    _excel_monthly = (_xls.set_index(date_col)[price_col]
                         .resample('ME').last().ffill())
    _excel_monthly.index = _excel_monthly.index.to_period('M').to_timestamp('M')
    # Sutura: Excel fino al 2014-12, poi FRED dal 2015-01
    _cutoff = pd.Timestamp('2014-12-31')
    djia_combined = pd.concat([
        _excel_monthly[_excel_monthly.index <= _cutoff],
        djia_fred[djia_fred.index > _cutoff]
    ]).sort_index()
    djia_combined.name = 'DJIA_Real'
    macro_df['DJIA_Real'] = djia_combined.reindex(macro_df.index)
    print(f"  DJIA combinato (Excel+FRED): {macro_df['DJIA_Real'].dropna().index[0].date()} – {macro_df['DJIA_Real'].dropna().index[-1].date()}")
    print(f"  Dati Excel disponibili: {(_xls[date_col].min().date())} – {(_xls[date_col].max().date())}")
except Exception as _e:
    print(f"  ⚠ Excel DJIA non trovato ({_e}), uso solo FRED")
    macro_df['DJIA_Real'] = djia_fred.reindex(macro_df.index)

# ── 9) Tenta download old GM (pre-fallimento 2009) da Stooq ────────────────
# Old GM (General Motors Corp.) fu delistata a giu-2009 dopo il fallimento.
# yfinance 'GM' copre solo nov-2010+. Tentiamo Stooq per coprire 2005-2009.
# Se non disponibile, il gap rimane e il DJIA proxy sovrastima quel periodo.
_gm_old = None
try:
    _gm_stooq = _download_stooq_monthly('gml.us')   # codice Stooq per old GM
    if _gm_stooq is None or _gm_stooq.empty:
        _gm_stooq = _download_stooq_monthly('gm.us')
    if _gm_stooq is not None and not _gm_stooq.empty:
        _gm_old = _gm_stooq[_gm_stooq.index < pd.Timestamp('2010-01-01')]
        if len(_gm_old) > 0:
            print(f"  ✓ Old GM (Stooq): {len(_gm_old)} mesi  [{_gm_old.index[0].date()} – {_gm_old.index[-1].date()}]")
        else:
            _gm_old = None
except Exception:
    _gm_old = None

if _gm_old is None:
    print("  ⚠ Old GM pre-2010 non disponibile — gap 2005-2009 rimane")
    print("    (il DJIA proxy sovrastima ~2-3% annuo in quel periodo)")

print(macro_df[['RF_Monthly_FRED','YieldCurve_10y2y','HY_Spread','DJIA_Real']].tail(6).round(5))


## 2c. VIX e Dollar Index (yfinance)

In [ ]:
# ── VIX: volatilità implicita S&P 500 ───────────────────────────────────────
vix_raw = yf.download('^VIX', start=START_DATE, end=END_DATE, interval='1mo',
                      auto_adjust=True, progress=False)
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)
vix_monthly = vix_raw['Close'].resample('ME').last()
vix_monthly.index = vix_monthly.index.to_period('M').to_timestamp('M')
vix_monthly.name = 'VIX'

# ── DXY: Dollar Index ────────────────────────────────────────────────────────
dxy_raw = yf.download('DX-Y.NYB', start=START_DATE, end=END_DATE, interval='1mo',
                      auto_adjust=True, progress=False)
if isinstance(dxy_raw.columns, pd.MultiIndex):
    dxy_raw.columns = dxy_raw.columns.get_level_values(0)
dxy_monthly = dxy_raw['Close'].resample('ME').last()
dxy_monthly.index = dxy_monthly.index.to_period('M').to_timestamp('M')
dxy_monthly.name = 'DXY'

# ── Aggiungi a macro_df ──────────────────────────────────────────────────────
macro_df['VIX'] = vix_monthly.reindex(macro_df.index)
macro_df['DXY'] = dxy_monthly.reindex(macro_df.index)

# VIX change MoM (segnale di risk-off)
macro_df['VIX_Chg'] = macro_df['VIX'].pct_change()
# DXY change MoM
macro_df['DXY_Chg'] = macro_df['DXY'].pct_change()

print(f"✓ macro_df aggiornato: {macro_df.shape}")
print(macro_df[['VIX','DXY','YieldCurve_10y2y','HY_Spread']].tail(6).round(3))


## 3. Backup embedded per WBA

Poiché `WBA` può risultare non scaricabile dalle fonti gratuite standard, qui incorporiamo direttamente una serie mensile di backup già pulita.  
In questo modo il notebook resta **self-contained** e replicabile anche senza file esterni.

In [3]:
# ── WBA embedded backup series (Investing, già pulita) ─────────────────────
# Questa serie è incorporata direttamente nel notebook per evitare dipendenze
# da file locali o fonti esterne per WBA.
WBA_EMBEDDED_CSV = '''\
Date,WBA
2018-06-01,60.015
2018-07-01,67.62
2018-08-01,68.56
2018-09-01,72.9
2018-10-01,79.77
2018-11-01,84.67
2018-12-01,68.33
2019-01-01,72.26
2019-02-01,71.19
2019-03-01,63.27
2019-04-01,53.57
2019-05-01,49.34
2019-06-01,54.67
2019-07-01,54.49
2019-08-01,51.19
2019-09-01,55.31
2019-10-01,54.78
2019-11-01,59.6
2019-12-01,58.96
2020-01-01,50.85
2020-02-01,45.76
2020-03-01,45.75
2020-04-01,43.29
2020-05-01,42.94
2020-06-01,42.39
2020-07-01,40.71
2020-08-01,38.02
2020-09-01,35.92
2020-10-01,34.04
2020-11-01,38.01
2020-12-01,39.88
2021-01-01,50.25
2021-02-01,47.93
2021-03-01,54.9
2021-04-01,53.1
2021-05-01,52.66
2021-06-01,52.61
2021-07-01,47.15
2021-08-01,50.75
2021-09-01,47.05
2021-10-01,47.02
2021-11-01,44.8
2021-12-01,52.16
2022-01-01,49.76
2022-02-01,46.09
2022-03-01,44.77
2022-04-01,42.4
2022-05-01,43.83
2022-06-01,37.9
2022-07-01,39.62
2022-08-01,35.06
2022-09-01,31.4
2022-10-01,36.5
2022-11-01,41.5
2022-12-01,37.36
2023-01-01,36.86
2023-02-01,35.53
2023-03-01,34.58
2023-04-01,35.25
2023-05-01,30.37
2023-06-01,28.49
2023-07-01,29.97
2023-08-01,25.31
2023-09-01,22.24
2023-10-01,21.08
2023-11-01,19.94
2023-12-01,26.11
2024-01-01,22.57
2024-02-01,21.26
'''


---
## 3. Composizione Storica DJI (Membership Point-in-Time)

Questa sezione definisce **chi era nell'indice e quando**.

- `ALL_DJIA_TICKERS`: i 49 ticker unici che hanno fatto parte del DJI dal 2005
- `TICKER_DOWNLOAD_MAP`: alias per i ticker rinominati/delistati (il download avviene sotto il nuovo nome, ma i dati vengono salvati sotto il ticker originale)
- `COMPOSITIONS`: la composizione esatta per ogni data di cambio
- `get_composition(date)` → restituisce i 30 ticker attivi in una data qualunque

> ⚠️ Il primo snapshot (21/11/2005) contiene 27 titoli invece di 30: il file sorgente originale era incompleto per quel periodo.


## 4. Universo storico dei ticker DJIA

Questa sezione definisce i **49 ticker unici** che sono entrati nel DJIA nel periodo analizzato, insieme alle eventuali mappature necessarie per il download.

In [4]:
# ── Lista completa dei 49 ticker storici ─────────────────────────────────────
ALL_DJIA_TICKERS = [
    'AA',   'AAPL', 'AIG',  'AMGN', 'AMZN', 'AXP',  'BA',   'BAC',
    'C',    'CAT',  'CRM',  'CSCO', 'CVX',  'DD',   'DIS',  'DOW',
    'DWDP', 'GE',   'GM',   'GS',   'HD',   'HON',  'HPQ',  'IBM',
    'INTC', 'JNJ',  'JPM',  'KFT',  'KO',   'MCD',  'MMM',  'MO',
    'MRK',  'MSFT', 'NKE',  'NVDA', 'PFE',  'PG',   'RTX',  'SHW',
    'T',    'TRV',  'UNH',  'UTX',  'V',    'VZ',   'WBA',  'WMT',
    'XOM'
]

# ── Alias ticker: composizione DJI → ticker Yahoo Finance ────────────────────
# Il dato viene scaricato sotto il ticker nuovo, ma salvato con il nome storico.
TICKER_DOWNLOAD_MAP = {
    'UTX':  'RTX',   # United Technologies → Raytheon (stessa storia su yf sotto RTX)
    'KFT':  'MDLZ',  # Kraft Foods → Mondelez (storia continua sotto MDLZ)
    'DWDP': 'DWDP',  # DowDuPont (Set 2017 – Apr 2019): dati limitati, fallback DOW+DD
    'WBA':  'WBA',   # Walgreens: delisted 2025, può richiedere fallback unadjusted
}

print(f"✓ Ticker totali: {len(ALL_DJIA_TICKERS)}")
print(f"  Alias definiti: {TICKER_DOWNLOAD_MAP}")

# ── Note su ticker con copertura storica parziale ────────────────────────────
# GM  : General Motors fallì nel 2009. Il nuovo GM (GMC) ricominciò a quotare
#        a nov 2010. yfinance 'GM' non ha dati pre-2010 → NaN 2005-2009
# UTX : United Technologies → si fuse con Raytheon in RTX (apr 2020).
#        yfinance 'RTX' ha storia retroattiva che copre anche il periodo UTX.
#        Se mancano dati pre-2020 scaricare prima 'UTX' come fallback.
# DWDP: DowDuPont esistette solo set 2017 – mar 2019, poi si spaccò in DOW/DD.
print("  GM: dati disponibili solo dal nov-2010 (nuovo GM dopo bancarotta 2009)")
print("  UTX → RTX: storia retroattiva disponibile su yfinance")


✓ Ticker totali: 49
  Alias definiti: {'UTX': 'RTX', 'KFT': 'MDLZ', 'DWDP': 'DWDP', 'WBA': 'WBA'}


## 5. Composizione point-in-time del DJIA

Qui viene definita la composizione storica dell'indice a ogni data di cambiamento.  
Questa parte è fondamentale perché evita bias di membership e permette una ricostruzione coerente nel tempo.

In [5]:
# ── Composizioni per ogni data di cambio ─────────────────────────────────────
CHANGE_DATES = [
    '2005-11-21','2008-02-19','2008-09-22','2009-06-08','2012-09-24',
    '2013-09-23','2015-03-19','2017-09-01','2018-06-26','2019-04-02',
    '2020-04-06','2020-08-31','2024-02-26','2024-11-08'
]

COMPOSITIONS = {
    '2005-11-21': ['AA','AIG','AXP','BA','C','CAT','DD','DIS','GE','GM','HD','HON',
                   'HPQ','IBM','INTC','JNJ','JPM','KO','MCD','MMM','MO','MRK','MSFT',
                   'PFE','PG','T','UTX','VZ','WMT','XOM'],            # 30 membri
    '2008-02-19': ['AA','AIG','AXP','BA','BAC','C','CAT','CVX','DD','DIS','GE','GM',
                   'HD','HPQ','IBM','INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT',
                   'PFE','PG','T','UTX','VZ','WMT','XOM'],
    '2008-09-22': ['AA','AXP','BA','BAC','C','CAT','CVX','DD','DIS','GE','GM','HD',
                   'HPQ','IBM','INTC','JNJ','JPM','KFT','KO','MCD','MMM','MRK','MSFT',
                   'PFE','PG','T','UTX','VZ','WMT','XOM'],
    '2009-06-08': ['AA','AXP','BA','BAC','CAT','CSCO','CVX','DD','DIS','GE','HD',
                   'HPQ','IBM','INTC','JNJ','JPM','KFT','KO','MCD','MMM','MRK','MSFT',
                   'PFE','PG','T','TRV','UTX','VZ','WMT','XOM'],
    '2012-09-24': ['AA','AXP','BA','BAC','CAT','CSCO','CVX','DD','DIS','GE','HD',
                   'HPQ','IBM','INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','PFE',
                   'PG','T','TRV','UNH','UTX','VZ','WMT','XOM'],
    '2013-09-23': ['AXP','BA','CAT','CSCO','CVX','DD','DIS','GE','GS','HD','IBM',
                   'INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE','PG',
                   'T','TRV','UNH','UTX','V','VZ','WMT','XOM'],
    '2015-03-19': ['AAPL','AXP','BA','CAT','CSCO','CVX','DD','DIS','GE','GS','HD',
                   'IBM','INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE',
                   'PG','TRV','UNH','UTX','V','VZ','WMT','XOM'],
    '2017-09-01': ['AAPL','AXP','BA','CAT','CSCO','CVX','DIS','DWDP','GE','GS','HD',
                   'IBM','INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE',
                   'PG','TRV','UNH','UTX','V','VZ','WMT','XOM'],
    '2018-06-26': ['AAPL','AXP','BA','CAT','CSCO','CVX','DIS','DWDP','GS','HD','IBM',
                   'INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE','PG',
                   'TRV','UNH','UTX','V','VZ','WBA','WMT','XOM'],
    '2019-04-02': ['AAPL','AXP','BA','CAT','CSCO','CVX','DIS','DOW','GS','HD','IBM',
                   'INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE','PG',
                   'TRV','UNH','UTX','V','VZ','WBA','WMT','XOM'],
    '2020-04-06': ['AAPL','AXP','BA','CAT','CSCO','CVX','DIS','DOW','GS','HD','IBM',
                   'INTC','JNJ','JPM','KO','MCD','MMM','MRK','MSFT','NKE','PFE','PG',
                   'RTX','TRV','UNH','V','VZ','WBA','WMT','XOM'],
    '2020-08-31': ['AAPL','AMGN','AXP','BA','CAT','CRM','CSCO','CVX','DIS','DOW',
                   'GS','HD','HON','IBM','INTC','JNJ','JPM','KO','MCD','MMM','MRK',
                   'MSFT','NKE','PG','TRV','UNH','V','VZ','WBA','WMT'],
    '2024-02-26': ['AAPL','AMGN','AMZN','AXP','BA','CAT','CRM','CSCO','CVX','DIS',
                   'DOW','GS','HD','HON','IBM','INTC','JNJ','JPM','KO','MCD','MMM',
                   'MRK','MSFT','NKE','PG','TRV','UNH','V','VZ','WMT'],
    '2024-11-08': ['AAPL','AMGN','AMZN','AXP','BA','CAT','CRM','CSCO','CVX','DIS',
                   'GS','HD','HON','IBM','JNJ','JPM','KO','MCD','MMM','MRK','MSFT',
                   'NKE','NVDA','PG','SHW','TRV','UNH','V','VZ','WMT'],
}

def get_composition(month_end):
    """Restituisce il set di ticker nell'indice per un dato mese.
    Applica backfill: usa il cambio composizione più recente ≤ month_end.
    """
    change_dates = sorted(pd.to_datetime(CHANGE_DATES))
    valid = [d for d in change_dates if d <= month_end]
    key = max(valid).strftime('%Y-%m-%d') if valid else '2005-11-21'
    return set(COMPOSITIONS[key])

# ── Verifica rapida ───────────────────────────────────────────────────────────
print("Date di cambio composizione:", len(CHANGE_DATES))
print("Composizioni registrate:    ", len(COMPOSITIONS))
print()
for dt in CHANGE_DATES:
    n = len(COMPOSITIONS[dt])
    flag = " ⚠ (incompleto)" if n < 30 else ""
    print(f"  {dt}  →  {n} titoli{flag}")


Date di cambio composizione: 14
Composizioni registrate:     14

  2005-11-21  →  30 titoli
  2008-02-19  →  30 titoli
  2008-09-22  →  30 titoli
  2009-06-08  →  30 titoli
  2012-09-24  →  30 titoli
  2013-09-23  →  30 titoli
  2015-03-19  →  30 titoli
  2017-09-01  →  30 titoli
  2018-06-26  →  30 titoli
  2019-04-02  →  30 titoli
  2020-04-06  →  30 titoli
  2020-08-31  →  30 titoli
  2024-02-26  →  30 titoli
  2024-11-08  →  30 titoli


## 5b. Configurazione Fondamentali

Costanti e funzioni helper per il download e il calcolo dei fondamentali trimestrali.

**Logica point-in-time**: per ogni mese usiamo solo i trimestri pubblicati
almeno **45 giorni prima** — replica la disponibilità reale dei dati, zero look-ahead bias.

**TTM (Trailing 12 Months)**: somma degli ultimi 4 trimestri disponibili per
Revenue, EBITDA, Net Income, EPS.


In [ ]:
# ── Mapping ticker → ticker yfinance per fondamentali ─────────────────────
# Serve per aziende riorganizzate/rinominate dopo la loro uscita dal DJIA
FUND_REMAP = {
    'UTX':  'RTX',    # United Technologies → Raytheon Technologies
    'KFT':  'MDLZ',   # Kraft Foods → Mondelez
    'DWDP': 'DOW',    # DowDuPont → Dow Inc.
    'AA':   'AA',
    'GM':   'GM',
    'HON':  'HON',
    'MO':   'MO',
    'T':    'T',
}

PUB_DELAY = pd.Timedelta(days=45)  # ritardo pubblicazione 10-Q (point-in-time)

# Colonne income statement cercate in ordine di preferenza
INC_COLS = {
    'Revenue':          ['Total Revenue', 'Operating Revenue'],
    'Gross_Profit':     ['Gross Profit'],
    'EBITDA':           ['EBITDA', 'Normalized EBITDA'],
    'Operating_Income': ['Operating Income', 'Operating Profit'],
    'Net_Income':       ['Net Income', 'Net Income Common Stockholders'],
    'EPS_Diluted':      ['Diluted EPS'],
}
# Colonne balance sheet cercate in ordine di preferenza
BS_COLS = {
    'Total_Equity':  ['Stockholders Equity', 'Total Equity Gross Minority Interest',
                      'Common Stock Equity', 'Total Stockholder Equity'],
    'Total_Debt':    ['Total Debt', 'Long Term Debt', 'Total Long Term Debt'],
    'Cash':          ['Cash And Cash Equivalents',
                      'Cash Cash Equivalents And Short Term Investments',
                      'Cash And Short Term Investments'],
    'Total_Assets':  ['Total Assets'],
    'Shares':        ['Ordinary Shares Number', 'Share Issued', 'Common Stock'],
    'Current_Assets':['Current Assets'],
    'Current_Liab':  ['Current Liabilities', 'Current Liabilities Total'],
}

# ── Helper functions ─────────────────────────────────────────────────────────
def get_val(row, candidates):
    """Primo valore non-NaN tra i candidati in una Series (riga di DataFrame)."""
    if row is None or (hasattr(row, 'empty') and row.empty):
        return np.nan
    for c in candidates:
        if c in row.index and pd.notna(row[c]):
            return float(row[c])
    return np.nan

def compute_ttm(df, col_candidates, n=4):
    """
    Trailing 12 Months: somma dei 4 trimestri piu recenti disponibili.
    df: gia filtrato point-in-time, righe = date trimestri.
    Restituisce NaN se meno di n trimestri disponibili.
    """
    for c in col_candidates:
        if c in df.columns:
            vals = df[c].dropna().sort_index(ascending=False)
            if len(vals) >= n:
                return float(vals.iloc[:n].sum())
    return np.nan

def _fetch_quarterly(ticker_symbol):
    """
    Scarica income statement e balance sheet trimestrali da yfinance.
    Ritorna (inc_T, bs_T) — DataFrame con righe=date, colonne=metriche — oppure (None, None).
    """
    try:
        t   = yf.Ticker(ticker_symbol)
        inc = t.quarterly_income_stmt
        bs  = t.quarterly_balance_sheet
        if inc is None or inc.empty:
            return None, None
        inc_T = inc.T.sort_index()
        bs_T  = bs.T.sort_index() if (bs is not None and not bs.empty) else pd.DataFrame()
        return inc_T, bs_T
    except Exception:
        return None, None

def _compute_pit_metrics(month, price, inc_T, bs_T):
    """
    Calcola tutte le metriche point-in-time per un (ticker, month).
    Ritorna dict con i valori oppure None se nessun dato trimestrale disponibile.
    """
    cutoff = pd.Timestamp(month) - PUB_DELAY

    inc_avail = inc_T[inc_T.index <= cutoff] if (inc_T is not None and not inc_T.empty) else pd.DataFrame()
    bs_avail  = bs_T[bs_T.index  <= cutoff] if (bs_T  is not None and not bs_T.empty)  else pd.DataFrame()

    if inc_avail.empty:
        return None

    # TTM da income statement
    rev_ttm    = compute_ttm(inc_avail, INC_COLS['Revenue'])
    gp_ttm     = compute_ttm(inc_avail, INC_COLS['Gross_Profit'])
    ebitda_ttm = compute_ttm(inc_avail, INC_COLS['EBITDA'])
    oi_ttm     = compute_ttm(inc_avail, INC_COLS['Operating_Income'])
    ni_ttm     = compute_ttm(inc_avail, INC_COLS['Net_Income'])
    eps_ttm    = compute_ttm(inc_avail, INC_COLS['EPS_Diluted'])

    # Balance sheet piu recente disponibile
    if not bs_avail.empty:
        bs_row = bs_avail.iloc[-1]
        equity = get_val(bs_row, BS_COLS['Total_Equity'])
        debt   = get_val(bs_row, BS_COLS['Total_Debt'])
        cash   = get_val(bs_row, BS_COLS['Cash'])
        assets = get_val(bs_row, BS_COLS['Total_Assets'])
        shares = get_val(bs_row, BS_COLS['Shares'])
        cur_a  = get_val(bs_row, BS_COLS['Current_Assets'])
        cur_l  = get_val(bs_row, BS_COLS['Current_Liab'])
    else:
        equity = debt = cash = assets = shares = cur_a = cur_l = np.nan

    p = float(price) if pd.notna(price) else np.nan
    mkt_cap = p * shares if (pd.notna(p) and pd.notna(shares) and shares > 0) else np.nan
    ev      = (mkt_cap + debt - cash) if all(pd.notna(x) for x in [mkt_cap, debt, cash]) else np.nan

    def r(a, b, pos_b=True):
        if pd.notna(a) and pd.notna(b) and (b > 0 if pos_b else b != 0):
            return a / b
        return np.nan

    return {
        'EPS_TTM':          eps_ttm,
        'Revenue_TTM':      rev_ttm,
        'EBITDA_TTM':       ebitda_ttm,
        'NetIncome_TTM':    ni_ttm,
        'PE_Ratio':         r(p,       eps_ttm),
        'PB_Ratio':         r(mkt_cap, equity),
        'PS_Ratio':         r(mkt_cap, rev_ttm),
        'EV_EBITDA':        r(ev,      ebitda_ttm),
        'ROE_TTM':          r(ni_ttm,  equity,  pos_b=False),
        'ROA_TTM':          r(ni_ttm,  assets),
        'Debt_Equity':      r(debt,    equity,  pos_b=False),
        'Gross_Margin':     r(gp_ttm,  rev_ttm),
        'Operating_Margin': r(oi_ttm,  rev_ttm),
        'Net_Margin':       r(ni_ttm,  rev_ttm),
        'Current_Ratio':    r(cur_a,   cur_l),
        'Total_Equity':     equity,
        'Total_Debt':       debt,
        'Cash':             cash,
    }

print("✓ Costanti e helper fondamentali pronti")
print(f"  INC_COLS: {list(INC_COLS)} | BS_COLS: {list(BS_COLS)}")


## 5c. Funzioni Helper per il Download Prezzi

Contiene `_try_download`, `_to_monthly_close`, `_clip_series_to_window`,
il patch multi-sorgente per WBA, e `TICKER_DOWNLOAD_MAP`.
Queste funzioni vengono usate all'interno del loop unificato nella sezione successiva.


In [ ]:
# ── Ticker alias map per download prezzi ────────────────────────────────
TICKER_DOWNLOAD_MAP = {
    'UTX':  'RTX',    # United Technologies → Raytheon
    'KFT':  'MDLZ',   # Kraft Foods → Mondelez
    'DWDP': 'DWDP',   # DowDuPont — fallback DOW+DD gestito nel loop
    'WBA':  'WBA',    # Walgreens — fallback embedded/Stooq gestito nel loop
}

def _try_download(ticker, start, end, auto_adjust=True):
    """Scarica dati da yfinance e restituisce un DataFrame OHLC o None."""
    try:
        data = yf.download(
            ticker,
            start=start,
            end=end,
            auto_adjust=auto_adjust,
            progress=False,
            threads=False,
        )
        if len(data) == 0:
            return None
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        return data
    except Exception:
        return None


def _read_csv_from_url(url):
    """Legge un CSV remoto con tentativi multipli di parsing."""
    read_attempts = [
        dict(),
        dict(sep=';'),
        dict(encoding='utf-8'),
    ]
    for kwargs in read_attempts:
        try:
            df = pd.read_csv(url, **kwargs)
            if isinstance(df, pd.DataFrame) and not df.empty:
                return df
        except Exception:
            pass
    return None


def _download_stooq_monthly(symbol):
    """Scarica una serie mensile da Stooq.

    Ritorna una Series mensile con indice datetime e nome 'Close'.
    Non richiede API key.
    """
    urls = [
        f"https://stooq.com/q/d/l/?s={symbol.lower()}&i=m",
        f"http://stooq.com/q/d/l/?s={symbol.lower()}&i=m",
    ]

    for url in urls:
        df = _read_csv_from_url(url)
        if df is None or df.empty:
            continue

        df.columns = [str(c).strip().replace('<', '').replace('>', '') for c in df.columns]

        if 'Date' not in df.columns:
            if len(df.columns) == 1:
                col = df.columns[0]
                if 'Date' in col and 'Close' in col:
                    tmp = df[col].str.split(',', expand=True)
                    header = [x.strip() for x in col.split(',')]
                    if tmp.shape[1] == len(header):
                        tmp.columns = header
                        df = tmp
            if 'Date' not in df.columns:
                continue

        close_col = None
        for c in ['Close', 'close']:
            if c in df.columns:
                close_col = c
                break
        if close_col is None:
            continue

        out = df[['Date', close_col]].copy()
        out['Date'] = pd.to_datetime(out['Date'], errors='coerce')
        out[close_col] = pd.to_numeric(out[close_col], errors='coerce')
        out = out.dropna(subset=['Date', close_col]).sort_values('Date')
        if out.empty:
            continue

        s = out.set_index('Date')[close_col]
        s.index = pd.to_datetime(s.index) + pd.offsets.MonthEnd(0)
        s = s[~s.index.duplicated(keep='last')].sort_index()
        s.name = 'Close'
        return s

    return None



def _load_embedded_wba():
    """Carica la serie WBA embedded nel notebook (fonte Investing già pulita).

    Output: Series mensile 'Close' con indice a fine mese.
    """
    try:
        df = pd.read_csv(StringIO(WBA_EMBEDDED_CSV))
        if df.empty:
            return None, None

        df.columns = [str(c).strip() for c in df.columns]
        if 'Date' not in df.columns:
            return None, None

        price_col = None
        for c in ['WBA', 'WBA_Close', 'Price', 'Close']:
            if c in df.columns:
                price_col = c
                break
        if price_col is None:
            return None, None

        out = df[['Date', price_col]].copy()
        out['Date'] = pd.to_datetime(out['Date'], errors='coerce')
        out[price_col] = pd.to_numeric(out[price_col], errors='coerce')
        out = out.dropna(subset=['Date', price_col]).sort_values('Date')
        if out.empty:
            return None, None

        s = out.set_index('Date')[price_col]
        s.index = pd.to_datetime(s.index) + pd.offsets.MonthEnd(0)
        s = s[~s.index.duplicated(keep='last')].sort_index()
        s.name = 'Close'
        return s, 'Embedded investing backup'
    except Exception:
        return None, None

def _load_local_wba_investing():
    """Carica la serie WBA pulita da file locale Investing.

    File attesi (primo disponibile):
    - WBA_clean_investing_full.csv
    - WBA_clean_price_series.csv

    Output: Series mensile 'Close' con indice a fine mese.
    """
    candidate_files = [
        Path("WBA_clean_investing_full.csv"),
        Path("WBA_clean_price_series.csv"),
    ]

    for fp in candidate_files:
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp)
            if df.empty:
                continue

            df.columns = [str(c).strip() for c in df.columns]

            date_col = 'Date' if 'Date' in df.columns else None
            if date_col is None:
                continue

            price_col = None
            for c in ['WBA_Close', 'WBA', 'Price', 'Close']:
                if c in df.columns:
                    price_col = c
                    break
            if price_col is None:
                continue

            out = df[[date_col, price_col]].copy()
            out[date_col] = pd.to_datetime(out[date_col], dayfirst=True, errors='coerce')
            out[price_col] = (
                out[price_col]
                .astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            out[price_col] = pd.to_numeric(out[price_col], errors='coerce')
            out = out.dropna(subset=[date_col, price_col]).sort_values(date_col)

            if out.empty:
                continue

            s = out.set_index(date_col)[price_col]
            # Il file Investing è mensile: allinea sempre a fine mese per combaciare con price_df
            s.index = pd.to_datetime(s.index) + pd.offsets.MonthEnd(0)
            s = s[~s.index.duplicated(keep='last')].sort_index()
            s.name = 'Close'
            return s, f"Local investing file: {fp.name}"
        except Exception:
            continue

    return None, None


def _to_monthly_close(data):
    """Converte un DataFrame OHLC giornaliero in una serie mensile di close."""
    if data is None or len(data) == 0:
        return None
    close = data['Close'] if 'Close' in data.columns else data.iloc[:, 0]
    monthly = close.resample('ME').last()
    monthly.name = 'Close'
    return monthly


def _clip_series_to_window(series, start, end):
    if series is None:
        return None
    start_ts = pd.to_datetime(start)
    end_ts = pd.to_datetime(end)
    series = series.sort_index()
    return series.loc[(series.index >= start_ts) & (series.index <= end_ts)]


def _download_wba_with_patch(start, end):
    """WBA: priorità serie embedded, poi file locale, poi Stooq, poi Yahoo.

    Ordine logico:
    1) serie embedded nel notebook (Investing già pulita)
    2) file locale Investing già pulito
    3) tenta Stooq
    4) tenta Yahoo adjusted
    5) tenta Yahoo non-adjusted
    6) merge con priorità alla fonte embedded/locale
    """
    embedded_series, embedded_note = _load_embedded_wba()
    embedded_series = _clip_series_to_window(embedded_series, start, end)

    local_series, local_note = _load_local_wba_investing()
    local_series = _clip_series_to_window(local_series, start, end)

    base_series = embedded_series
    base_note = embedded_note
    if (base_series is None or len(base_series) == 0) and local_series is not None and len(local_series) > 0:
        base_series = local_series
        base_note = local_note

    stooq_monthly = _clip_series_to_window(_download_stooq_monthly('wba.us'), start, end)

    yf_data = _try_download('WBA', start, end, auto_adjust=True)
    if yf_data is None:
        yf_data = _try_download('WBA', start, end, auto_adjust=False)
    yf_monthly = _clip_series_to_window(_to_monthly_close(yf_data), start, end)

    sources = []

    if base_series is not None and len(base_series) > 0:
        combined = base_series.copy()
        sources.append(base_note)

        # Se esiste anche un file locale distinto, usalo come patch secondaria
        if local_series is not None and len(local_series) > 0 and base_note != local_note:
            before = int(combined.notna().sum())
            combined = combined.combine_first(local_series)
            after = int(combined.notna().sum())
            sources.append(f"Local patch (+{after - before} mesi)")

        if yf_monthly is not None and len(yf_monthly) > 0:
            before = int(combined.notna().sum())
            combined = combined.combine_first(yf_monthly)
            after = int(combined.notna().sum())
            sources.append(f"Yahoo patch (+{after - before} mesi)")

        if stooq_monthly is not None and len(stooq_monthly) > 0:
            before = int(combined.notna().sum())
            combined = combined.combine_first(stooq_monthly)
            after = int(combined.notna().sum())
            sources.append(f"Stooq patch (+{after - before} mesi)")

        return combined, " | ".join(sources)

    if stooq_monthly is None and yf_monthly is None:
        return None, "FAILED: no embedded/local WBA source, Yahoo e Stooq non disponibili"

    if stooq_monthly is not None and yf_monthly is None:
        return stooq_monthly, "Stooq only"

    if stooq_monthly is None and yf_monthly is not None:
        return yf_monthly, "Yahoo only"

    combined = yf_monthly.combine_first(stooq_monthly)
    n_yf = int(yf_monthly.notna().sum())
    n_stooq = int(stooq_monthly.notna().sum())
    n_combined = int(combined.notna().sum())

    if n_combined > n_yf:
        return combined, f"Yahoo + Stooq patch (+{n_combined - n_yf} mesi; Stooq={n_stooq})"
    return combined, f"Yahoo only (Stooq checked; Stooq={n_stooq})"

print('✓ Helper prezzi pronti (TICKER_DOWNLOAD_MAP, _try_download, WBA patch, ecc.)')


## 6. Download Prezzi Mensili

Scarica i prezzi adjusted mensili per tutti i 49 ticker storici.
Gestione speciale per **WBA** (multi-sorgente) e **DWDP** (fallback DOW+DD).
Output: `price_df` (matrice wide mesi × ticker) e `download_metadata`.


In [ ]:
# ── Download prezzi mensili per tutti i ticker ──────────────────────────────

prices        = {}
metadata_rows = []
failed_price  = []

print(f"Download prezzi: {len(ALL_DJIA_TICKERS)} ticker  ({START_DATE} → {END_DATE})")
print("-" * 60)

for tk in ALL_DJIA_TICKERS:
    yf_tk    = TICKER_DOWNLOAD_MAP.get(tk, tk)
    monthly  = None
    src_note = "Yahoo"

    if tk == 'WBA':
        monthly, src_note = _download_wba_with_patch(START_DATE, END_DATE)

    elif tk == 'DWDP':
        data = _try_download('DWDP', start='2017-09-01', end='2019-05-01')
        monthly = _to_monthly_close(data)
        if monthly is None:
            dow = _to_monthly_close(_try_download('DOW', '2017-09-01', '2019-05-01'))
            dd  = _to_monthly_close(_try_download('DD',  '2017-09-01', '2019-05-01'))
            if dow is not None and dd is not None:
                monthly  = pd.concat([dow.rename('DOW'), dd.rename('DD')], axis=1).mean(axis=1)
                monthly.name = 'Close'
                src_note = 'Fallback media DOW/DD'

    else:
        data    = _try_download(yf_tk, START_DATE, END_DATE)
        monthly = _to_monthly_close(data)
        if yf_tk != tk and monthly is not None:
            src_note = f"Yahoo via {yf_tk}"

    monthly = _clip_series_to_window(monthly, START_DATE, END_DATE)

    n_m = 0
    if monthly is not None and len(monthly) > 0:
        prices[tk] = monthly
        n_m = int(monthly.notna().sum())
        print(f"  ✓ {tk:<6}  {n_m:>3} mesi  [{src_note}]")
    else:
        failed_price.append(tk)
        print(f"  ✗ {tk:<6}  NO DATA  [{src_note}]")

    metadata_rows.append({
        'Ticker':     tk,
        'Months':     n_m,
        'Source':     src_note,
        'First_Date': monthly.dropna().index.min() if n_m > 0 else pd.NaT,
        'Last_Date':  monthly.dropna().index.max() if n_m > 0 else pd.NaT,
    })

# Assembla price_df
price_df = pd.DataFrame(prices)
price_df.index = pd.to_datetime(price_df.index)
ordered_cols = [t for t in ALL_DJIA_TICKERS if t in price_df.columns]
price_df = price_df[ordered_cols]

download_metadata = pd.DataFrame(metadata_rows).sort_values('Ticker').reset_index(drop=True)

print("-" * 60)
print(f"\n✓ price_df: {price_df.shape[0]} mesi x {price_df.shape[1]} ticker")
if failed_price:
    print(f"⚠ Senza prezzi ({len(failed_price)}): {failed_price}")


# ── Patch old GM pre-2010 (se disponibile da Stooq) ─────────────────────────
if '_gm_old' in globals() and _gm_old is not None and len(_gm_old) > 0:
    _gm_old.index = pd.to_datetime(_gm_old.index) + pd.offsets.MonthEnd(0)
    if 'GM' in price_df.columns:
        # Riempi solo i NaN pre-2010
        _mask = price_df.index < pd.Timestamp('2010-01-01')
        price_df.loc[_mask, 'GM'] = price_df.loc[_mask, 'GM'].fillna(_gm_old.reindex(price_df.index[_mask]))
        n_filled = price_df.loc[_mask, 'GM'].notna().sum()
        print(f"  ✓ GM patch applicata: {n_filled} mesi pre-2010 riempiti con dati Stooq")
    else:
        price_df['GM'] = _gm_old.reindex(price_df.index)
        print(f"  ✓ GM aggiunto da Stooq: {_gm_old.notna().sum()} mesi")
else:
    print("  ~ GM pre-2010: nessun dato Stooq disponibile, gap 2005-2009 confermato")


## 6b. Download Fondamentali Trimestrali

Per ogni ticker scarica **income statement** e **balance sheet** trimestrali
(una sola chiamata yfinance per ticker).

I dati vengono conservati grezzi in `fund_quarterly` — dizionario
`{ticker: {'inc': DataFrame, 'bs': DataFrame}}`.
Il calcolo delle metriche avviene dopo, dentro `build_panel`, mese per mese.


In [ ]:
# ── Download fondamentali trimestrali (una chiamata per ticker) ──────────────

fund_quarterly = {}
no_fund = []

print(f"Download fondamentali trimestrali: {len(ALL_DJIA_TICKERS)} ticker")
print("(puo richiedere 3-5 minuti)")
print("-" * 60)

for tk in ALL_DJIA_TICKERS:
    dl_tk = FUND_REMAP.get(tk, tk)
    inc_T, bs_T = _fetch_quarterly(dl_tk)
    if inc_T is not None:
        fund_quarterly[tk] = {'inc': inc_T, 'bs': bs_T, 'dl_ticker': dl_tk}
        n_q = len(inc_T)
        dt0 = inc_T.index.min().strftime('%Y-%m') if len(inc_T) > 0 else '?'
        dt1 = inc_T.index.max().strftime('%Y-%m') if len(inc_T) > 0 else '?'
        print(f"  ✓ {tk:<6}  {n_q:>2} trimestri  [{dt0} → {dt1}]  (scaricato come {dl_tk})")
    else:
        fund_quarterly[tk] = None
        no_fund.append(tk)
        print(f"  ✗ {tk:<6}  NO DATA")

print("-" * 60)
print(f"\n✓ Con fondamentali: {len(fund_quarterly) - len(no_fund)} / {len(ALL_DJIA_TICKERS)}")
if no_fund:
    print(f"⚠ Senza fondamentali ({len(no_fund)}): {no_fund}")


---
## 5. Salvataggio Matrice Prezzi

Salva la matrice grezza `(mesi × ticker)` come CSV.  
Ogni colonna = un ticker, ogni riga = un mese (fine mese).  
I valori mancanti (`NaN`) indicano che il ticker non era ancora quotato o era già delisted.


## 7. Salvataggio della price matrix

Qui esportiamo la matrice prezzi in formato wide:  
- righe = mesi  
- colonne = ticker  
- valori = prezzi di fine mese

In [7]:
# Salva matrice prezzi grezza
price_df.to_csv('price_matrix_all49.csv')
print("✓ Salvato: price_matrix_all49.csv")
print(f"  Shape: {price_df.shape}")
print(f"  Periodo: {price_df.index[0].date()}  →  {price_df.index[-1].date()}")
print()

# Preview: quanti mesi di dati per ticker
coverage = price_df.notna().sum().sort_values(ascending=False)
print("Copertura per ticker (mesi con dati):")
print(coverage.to_string())


✓ Salvato: price_matrix_all49.csv
  Shape: (256, 49)
  Periodo: 2005-01-31  →  2026-04-30

Copertura per ticker (mesi con dati):
AA      256
AAPL    256
AIG     256
AMGN    256
AMZN    256
AXP     256
BA      256
BAC     256
C       256
CAT     256
CRM     256
CSCO    256
CVX     256
DD      256
DIS     256
GS      256
GE      256
TRV     256
NKE     256
HD      256
HON     256
IBM     256
HPQ     256
INTC    256
JNJ     256
MCD     256
JPM     256
KFT     256
KO      256
MO      256
MMM     256
MRK     256
MSFT    256
XOM     256
UNH     256
NVDA    256
PFE     256
PG      256
RTX     256
SHW     256
T       256
WMT     256
VZ      256
UTX     256
V       218
GM      186
DOW      86
WBA      69
DWDP     20


---
## 6. Costruzione Panel Long-Format

Trasforma la matrice wide `(mesi × ticker)` in un panel long `(mesi × ticker, una riga per combinazione)`.

Ogni riga contiene:
| Colonna | Descrizione |
|---|---|
| `Date` | Fine mese |
| `Ticker` | Ticker del titolo |
| `In_Index` | 1 se il titolo è membro del DJIA quel mese, 0 altrimenti |
| `Price` | Prezzo adjusted a fine mese |
| `Return_monthly` | Rendimento mensile (Price_t / Price_{t-1} − 1) |
| `Price_Available` | 1 se il prezzo del mese è disponibile, 0 altrimenti |
| `Include_In_Training` | 1 se il titolo è eligibile nel sotto-universo clean di quel mese |
| `Missing_In_Index` | 1 se il titolo è in indice ma senza prezzo |
| `Missing_Reason` | motivazione esplicita del missing (es. GM pre-bankruptcy / unavailable) |
| `Weight_PW` | Peso price-weighted storico calcolato sui prezzi disponibili |
| `Weight_PW_Clean` | Peso price-weighted calcolato solo sul sotto-universo eligibile |
| `Weight_EW` | Peso equal-weight teorico 1/30 per i membri del DJIA |
| `Weight_EW_Clean` | Equal-weight del sotto-universo eligibile del mese |
| `RF_Monthly` | Risk-free mensile |
| `Sum_Prices_InIndex` | Somma prezzi dei membri in indice con prezzo disponibile |
| `Sum_Prices_Eligible` | Somma prezzi dei membri eligibili nel sotto-universo clean |


## 8. Costruzione del panel long-format

In questa sezione trasformiamo la price matrix in un dataset panel, una riga per ogni combinazione `Date × Ticker`.

Il panel contiene:
- membership nell'indice
- prezzo
- rendimento mensile
- flag di disponibilità del prezzo
- pesi clean per benchmark e training

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Forward-fill prezzi (per denominatore pesi e chain-linked divisor)
# ─────────────────────────────────────────────────────────────────────────────
price_ffill = price_df.ffill()

# ── FORMULA PESI — spiegazione esplicita ─────────────────────────────────────
#
#  Il DJIA è un indice PRICE-WEIGHTED. Il peso di ogni titolo è semplicemente:
#
#       W_i(t) = P_i(t) / Σ_j P_j(t)    per j ∈ composizione DJIA a t
#
#  Il rendimento mensile dell'indice è:
#
#       R_DJIA(t) = Σ_i [ W_i(t-1) × r_i(t) ]
#
#  CHIAVE: si usano i pesi del MESE PRECEDENTE (t-1), non del mese corrente.
#  Questo corrisponde a tenere la composizione fissa per tutto il mese e
#  rivalutarla a fine mese con i nuovi prezzi.
#
#  Nel panel:
#    Weight_PW_begin = P_i(t-1) / Σ_j P_j(t-1)  ← usato per attribuire R(t)
#    Weight_PW_end   = P_i(t)   / Σ_j P_j(t)    ← posizione fine mese
#
#  Per entrambi usiamo price_ffill al denominatore:
#  se un titolo in-index non ha prezzo quel mese (es. GM 2005-2009),
#  usiamo l'ultimo prezzo noto → i 30 titoli pesano correttamente.
# ─────────────────────────────────────────────────────────────────────────────

# ── Chain-linked divisor per DJIA_Proxy ──────────────────────────────────────
def build_shadow_divisor(price_ffill, change_dates, djia_real_s=None):
    """
    D(t): divisore chain-linked inizializzato dal DJIA reale.
    Formula: DJIA_proxy(t) = Σ_i P_i(t) / D(t)
    Aggiornamento a ogni cambio composizione:
        D_new = D_old × Σ(prezzi nuova comp) / Σ(prezzi vecchia comp)
    """
    months     = sorted(price_ffill.index)
    change_ts  = {pd.to_datetime(cd) + pd.offsets.MonthEnd(0) for cd in change_dates}
    D, divisors = None, {}

    for month in months:
        comp_now = get_composition(month)
        sum_now  = sum(price_ffill.loc[month, tk]
                       for tk in comp_now
                       if tk in price_ffill.columns and pd.notna(price_ffill.loc[month, tk]))

        if D is None:
            if djia_real_s is not None and month in djia_real_s.index:
                v = djia_real_s.loc[month]
                if pd.notna(v) and v > 0 and sum_now > 0:
                    D = sum_now / v
            if D is None:
                divisors[month] = np.nan
                continue

        if month in change_ts:
            prev = month - pd.DateOffset(months=1)
            comp_prev = get_composition(prev)
            sum_prev  = sum(price_ffill.loc[month, tk]
                            for tk in comp_prev
                            if tk in price_ffill.columns and pd.notna(price_ffill.loc[month, tk]))
            if sum_prev > 0 and sum_now > 0:
                D = D * sum_now / sum_prev

        divisors[month] = D
    return pd.Series(divisors, name='DJIA_Divisor')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Build panel — due DataFrame separati
# ─────────────────────────────────────────────────────────────────────────────
def build_panels(price_df, price_ffill, months, fund_quarterly=None,
                 change_dates=None, djia_real_s=None):
    """
    Restituisce (panel_returns, panel_valuation).

    panel_returns   : una riga per (mese, ticker)
                      prezzi, rendimenti, pesi lagged+correnti, membership
    panel_valuation : una riga per (mese, ticker) SOLO dove dati fondamentali
                      esistono — PE, PB, PS, EV/EBITDA, margini, ecc.
    """
    months_list = sorted(price_df.index)
    all_tickers = sorted(price_df.columns)

    djia_div = build_shadow_divisor(price_ffill, change_dates or [], djia_real_s)

    returns_rows    = []
    valuation_rows  = []

    for month in months:
        if month not in price_df.index:
            continue

        comp     = get_composition(month)
        month_pos = months_list.index(month)
        prev_month = months_list[month_pos - 1] if month_pos > 0 else None

        # ── RF mensile ────────────────────────────────────────────────────────
        if 'rf_monthly_fred' in globals() and not rf_monthly_fred.empty:
            mts  = pd.Timestamp(month).to_period('M').to_timestamp('M')
            ridx = rf_monthly_fred.index[rf_monthly_fred.index <= mts]
            rf   = float(rf_monthly_fred.loc[ridx[-1]]) if len(ridx) > 0 else 0.0
        else:
            rf = 0.0

        # ── Denominatori per i pesi ───────────────────────────────────────────
        # END-of-month: usa prezzi ffill del mese corrente (t)
        sum_end = sum(price_ffill.loc[month, tk]
                      for tk in comp
                      if tk in price_ffill.columns and pd.notna(price_ffill.loc[month, tk]))

        # BEGIN-of-month (lagged): usa prezzi ffill del mese precedente (t-1)
        # → questi sono i pesi che moltiplicano il rendimento di t per R_DJIA(t)
        if prev_month is not None:
            comp_prev = get_composition(prev_month)
            sum_begin = sum(price_ffill.loc[prev_month, tk]
                            for tk in comp_prev
                            if tk in price_ffill.columns and pd.notna(price_ffill.loc[prev_month, tk]))
        else:
            comp_prev, sum_begin = comp, sum_end

        # ── Eligible (prezzo reale disponibile) ──────────────────────────────
        eligible = [tk for tk in comp
                    if tk in price_df.columns and pd.notna(price_df.loc[month, tk])]
        n_elig   = len(eligible)
        sum_real = sum(price_df.loc[month, tk] for tk in eligible)

        # ── DJIA proxy via divisore ───────────────────────────────────────────
        djia_proxy_m = np.nan
        if month in djia_div.index and pd.notna(djia_div.loc[month]) and djia_div.loc[month] > 0:
            djia_proxy_m = sum_end / djia_div.loc[month]

        # ── Loop per ticker ───────────────────────────────────────────────────
        for tk in all_tickers:
            in_idx = 1 if tk in comp else 0
            price  = price_df.loc[month, tk]  if tk in price_df.columns  else np.nan
            price_ff = price_ffill.loc[month, tk] if tk in price_ffill.columns else np.nan
            price_ff_prev = (price_ffill.loc[prev_month, tk]
                             if prev_month is not None and tk in price_ffill.columns
                             else np.nan)

            # Rendimento mensile
            ret = np.nan
            if prev_month is not None:
                prev_p = price_df.loc[prev_month, tk] if tk in price_df.columns else np.nan
                if pd.notna(price) and pd.notna(prev_p) and prev_p != 0:
                    ret = price / prev_p - 1

            missing_in_idx = int(in_idx == 1 and pd.isna(price))
            if missing_in_idx:
                miss_r = ('GM: nuovo IPO nov-2010, no dati pre-2010' if tk=='GM'
                           else 'DWDP: set-2017 / mar-2019 soltanto'  if tk=='DWDP'
                           else 'Prezzo non disponibile')
            else:
                miss_r = ''

            include = int(in_idx == 1 and tk in eligible)

            # ── Pesi ─────────────────────────────────────────────────────────
            # Weight_PW_begin: peso lagged (t-1) → corretto per R_DJIA(t)
            in_prev = 1 if tk in comp_prev else 0
            w_begin = (price_ff_prev / sum_begin
                       if in_prev == 1 and pd.notna(price_ff_prev) and sum_begin > 0
                       else np.nan)
            # Weight_PW_end: peso corrente (t)
            w_end   = (price_ff / sum_end
                       if in_idx == 1 and pd.notna(price_ff) and sum_end > 0
                       else np.nan)
            # Equal-weight
            w_ew       = 1/len(comp)      if in_idx==1 and len(comp)>0    else np.nan
            w_ew_clean = 1/n_elig         if include==1 and n_elig>0      else np.nan

            returns_rows.append({
                'Date':             month,
                'Ticker':           tk,
                'In_Index':         in_idx,
                'Price':            round(float(price), 4)   if pd.notna(price) else np.nan,
                'Return_monthly':   round(float(ret), 6)     if pd.notna(ret)   else np.nan,
                'RF_Monthly':       round(rf, 6),
                'Price_Available':  int(pd.notna(price)),
                'Include_Training': include,
                'Missing_In_Index': missing_in_idx,
                'Missing_Reason':   miss_r,
                # Peso lagged (t-1): usa per attribuire rendimento t all'indice
                'Weight_PW_begin':  round(float(w_begin), 6) if pd.notna(w_begin) else np.nan,
                # Peso corrente (t): posizione fine mese
                'Weight_PW_end':    round(float(w_end),   6) if pd.notna(w_end)   else np.nan,
                'Weight_EW':        round(float(w_ew),    6) if pd.notna(w_ew)    else np.nan,
                'Weight_EW_clean':  round(float(w_ew_clean),6) if pd.notna(w_ew_clean) else np.nan,
                'N_Eligible':       n_elig if in_idx==1 else np.nan,
                'DJIA_Proxy':       round(float(djia_proxy_m),2) if pd.notna(djia_proxy_m) else np.nan,
            })

            # ── Valuation (solo se dati fondamentali disponibili) ─────────────
            fdata = (fund_quarterly or {}).get(tk)
            if fdata is not None:
                val = _compute_pit_metrics(month, price, fdata['inc'], fdata['bs']) or {}
                if val:
                    valuation_rows.append({
                        'Date':             month,
                        'Ticker':           tk,
                        'Price':            round(float(price),4) if pd.notna(price) else np.nan,
                        'EPS_TTM':          val.get('EPS_TTM'),
                        'Revenue_TTM':      val.get('Revenue_TTM'),
                        'EBITDA_TTM':       val.get('EBITDA_TTM'),
                        'PE_Ratio':         round(val['PE_Ratio'],4)         if pd.notna(val.get('PE_Ratio'))   else np.nan,
                        'PB_Ratio':         round(val['PB_Ratio'],4)         if pd.notna(val.get('PB_Ratio'))   else np.nan,
                        'PS_Ratio':         round(val['PS_Ratio'],4)         if pd.notna(val.get('PS_Ratio'))   else np.nan,
                        'EV_EBITDA':        round(val['EV_EBITDA'],4)        if pd.notna(val.get('EV_EBITDA'))  else np.nan,
                        'ROE_TTM':          round(val['ROE_TTM'],4)          if pd.notna(val.get('ROE_TTM'))    else np.nan,
                        'ROA_TTM':          round(val['ROA_TTM'],4)          if pd.notna(val.get('ROA_TTM'))    else np.nan,
                        'Debt_Equity':      round(val['Debt_Equity'],4)      if pd.notna(val.get('Debt_Equity')) else np.nan,
                        'Gross_Margin':     round(val['Gross_Margin'],4)     if pd.notna(val.get('Gross_Margin')) else np.nan,
                        'Operating_Margin': round(val['Operating_Margin'],4) if pd.notna(val.get('Operating_Margin')) else np.nan,
                        'Net_Margin':       round(val['Net_Margin'],4)       if pd.notna(val.get('Net_Margin')) else np.nan,
                        'Current_Ratio':    round(val['Current_Ratio'],4)    if pd.notna(val.get('Current_Ratio')) else np.nan,
                    })

    panel_ret = pd.DataFrame(returns_rows)
    panel_val = pd.DataFrame(valuation_rows)
    if not panel_ret.empty:
        panel_ret['Date'] = pd.to_datetime(panel_ret['Date'])
    if not panel_val.empty:
        panel_val['Date'] = pd.to_datetime(panel_val['Date'])
    return panel_ret, panel_val


# ── Esegui ───────────────────────────────────────────────────────────────────
djia_real_s = None
if 'macro_df' in globals() and 'DJIA_Real' in macro_df.columns:
    djia_real_s = macro_df['DJIA_Real'].dropna()
    djia_real_s.index = pd.to_datetime(djia_real_s.index) + pd.offsets.MonthEnd(0)

panel_returns, panel_valuation = build_panels(
    price_df, price_ffill,
    months       = price_df.index,
    fund_quarterly  = fund_quarterly,
    change_dates    = CHANGE_DATES,
    djia_real_s     = djia_real_s,
)

# Mantieni anche 'panel' come alias per compatibilità con le celle rolling/export
panel = panel_returns.copy()

print(f"✓ panel_returns:   {panel_returns.shape[0]:,} righe x {panel_returns.shape[1]} colonne")
print(f"✓ panel_valuation: {panel_valuation.shape[0]:,} righe x {panel_valuation.shape[1]} colonne")
print(f"  Ticker con valutazioni: {panel_valuation['Ticker'].nunique() if not panel_valuation.empty else 0}")
print()

# ── Esempio esplicito calcolo pesi ───────────────────────────────────────────
# Mostra Weight_PW_begin e Weight_PW_end per un mese di esempio
ex_month = panel_returns['Date'].dropna().sort_values().iloc[-1]   # ultimo mese
sub_ex = panel_returns[
    (panel_returns['Date']==ex_month) &
    (panel_returns['In_Index']==1) &
    panel_returns['Weight_PW_begin'].notna()
].sort_values('Weight_PW_begin', ascending=False)

print("=" * 70)
print(f"ESEMPIO CALCOLO PESI — {str(ex_month)[:10]}")
print("=" * 70)
print(f"Formula: W_i(t-1) = P_i(t-1) / Σ_j P_j(t-1)  per j ∈ DJIA(t-1)")
print(f"         Denominatore Σ P_j(t-1) = {sub_ex['Weight_PW_begin'].sum():.6f} (deve essere ≈ 1.0)")
print()
print(f"  {'Ticker':<6} {'Prezzo t-1':>11} {'W_begin (lagged)':>17} {'Prezzo t':>10} {'W_end (corrente)':>17} {'Rendim.':>9}")
print("  " + "-"*78)
for _, r in sub_ex.iterrows():
    w_end = r['Weight_PW_end'] if pd.notna(r['Weight_PW_end']) else float('nan')
    ret   = r['Return_monthly'] if pd.notna(r['Return_monthly']) else float('nan')
    # Stima P(t-1) da peso e denominatore
    p_now = r['Price'] if pd.notna(r['Price']) else float('nan')
    print(f"  {r['Ticker']:<6} {p_now:>11.2f} {r['Weight_PW_begin']:>17.4%} {p_now:>10.2f} {w_end:>17.4%} {ret:>9.2%}")
print("  " + "-"*78)
print(f"  {'TOTALE':<6} {'':>11} {sub_ex['Weight_PW_begin'].sum():>17.4%}")
print()
print("DJIA_return(t) = Σ [Weight_PW_begin × Return_monthly] =",
      f"{(sub_ex['Weight_PW_begin'] * sub_ex['Return_monthly'].fillna(0)).sum():.4%}")


## 8b. Valuation nel Panel

Le metriche di valutazione (PE, PB, PS, EV/EBITDA, ROE, ROA, Margini, Current Ratio)
sono già integrate nel panel — calcolate direttamente dentro `build_panel` per ogni
riga `(mese, ticker)` usando il prezzo di mercato del mese e i fondamentali trimestrali
disponibili con 45 giorni di ritardo.


In [ ]:
# ── Copertura metriche valutazione ──────────────────────────────────────────
val_cols = ['PE_Ratio','PB_Ratio','PS_Ratio','EV_EBITDA',
            'ROE_TTM','ROA_TTM','Debt_Equity','Net_Margin','Current_Ratio']

print("=" * 60)
print("COPERTURA VALUATION NEL PANEL (righe In_Index=1)")
print("=" * 60)
sub = panel[panel['In_Index'] == 1]
for c in val_cols:
    if c in panel.columns:
        n   = sub[c].notna().sum()
        pct = 100 * n / len(sub) if len(sub) > 0 else 0
        bar = '#' * int(pct // 5)
        print(f"  {c:<22} {pct:5.1f}%  {bar}")

# DJIA Proxy vs reale
print()
if 'DJIA_Proxy' in panel.columns and panel['DJIA_Proxy'].notna().any():
    proxy_series = (panel.groupby('Date')['DJIA_Proxy'].first()
                    .dropna().sort_index())
    print(f"DJIA Proxy (chain-linked): {len(proxy_series)} mesi disponibili")
    if 'macro_df' in globals() and 'DJIA_Real' in macro_df.columns:
        real = macro_df['DJIA_Real'].reindex(proxy_series.index).dropna()
        common = proxy_series.reindex(real.index).dropna()
        if len(common) > 0:
            corr = common.corr(real.loc[common.index])
            te   = (common.pct_change() - real.loc[common.index].pct_change()).std() * (12**0.5) * 100
            print(f"  Correlazione vs DJIA reale : {corr:.4f}")
            print(f"  Tracking Error annualizzato: {te:.2f}%")


---
## 7. Calcolo Indicatori Rolling

Aggiunge al panel due indicatori calcolati su finestra mobile di **12 mesi**:

- **`Sharpe_12m`**: Sharpe Ratio annualizzato rolling = `(mean(excess_ret) × 12) / (std(ret) × √12)`
  - `excess_ret = Return_monthly − RF_Monthly`
  - richiede almeno 12 osservazioni valide
- **`Volatility_12m`**: deviazione standard annualizzata dei rendimenti mensili = `std × √12`

Il calcolo viene fatto per ogni ticker separatamente con `groupby('Ticker')`.


## 9. Indicatori rolling a 12 mesi

Qui aggiungiamo indicatori mobili utili per l'analisi quantitativa, come:
- **Sharpe ratio rolling**
- **massimo drawdown rolling**

Questi campi sono utili sia per diagnostica sia come possibili feature per il modello.

In [9]:
def sharpe_12m(group):
    """Sharpe Ratio annualizzato su rolling 12 mesi.
    
    Nota RF:
    - RF_Monthly nel panel = TB3MS_FRED / 100 / 12 → tasso mensile reale
    - La sottrazione excess = ret - RF_mensile è corretta
    - Fonte: Federal Reserve FRED, serie TB3MS
    """
    excess = group['Return_monthly'] - group['RF_Monthly']
    roll_mean = excess.rolling(12, min_periods=12).mean()
    roll_std  = group['Return_monthly'].rolling(12, min_periods=12).std()
    return (roll_mean * 12) / (roll_std * np.sqrt(12))


panel = panel.sort_values(['Ticker', 'Date']).reset_index(drop=True)

panel['Sharpe_12m'] = (
    panel.groupby('Ticker', group_keys=False)
         .apply(sharpe_12m)
         .reset_index(level=0, drop=True)
)

panel['Volatility_12m'] = (
    panel.groupby('Ticker', group_keys=False)['Return_monthly']
         .transform(lambda x: x.rolling(12, min_periods=12).std() * np.sqrt(12))
)

print("✓ Indicatori rolling aggiunti: Sharpe_12m, Volatility_12m")
print("  (RF usato: FRED TB3MS / 100 / 12, tasso mensile reale)")


✓ Indicatori rolling aggiunti: Sharpe_12m, Volatility_12m

Esempio — MSFT (ultimi 12 mesi):
      Date    Price  Return_monthly  Weight_PW  Weight_PW_Clean  Weight_EW_Clean  Sharpe_12m  Volatility_12m
2025-05-31 457.7011        0.166840   0.067627         0.067627         0.033333    0.370754        0.233718
2025-06-30 494.5372        0.080481   0.069940         0.069940         0.033333    0.388719        0.234915
2025-07-31 530.4188        0.072556   0.074874         0.074874         0.033333    1.009639        0.226512
2025-08-31 504.5917       -0.048692   0.068876         0.068876         0.033333    0.768295        0.236800
2025-09-30 515.8051        0.022223   0.069021         0.069021         0.033333    0.733537        0.236445
2025-10-31 515.6656       -0.000270   0.067250         0.067250         0.033333    1.026648        0.223703
2025-11-30 490.8896       -0.048047   0.063725         0.063725         0.033333    0.593556        0.232934
2025-12-31 482.5187       -0.017053 

---
## 8. Export CSV

Salva il panel finale. I file vengono scritti nella directory corrente di Colab (`/content/`).

Per scaricarli sul tuo computer:
```python
from google.colab import files
files.download('/content/panel_dataset_all49_2005_2026.csv')
files.download('/content/price_matrix_all49.csv')
```


## 10. Export finale dei file

Questa sezione salva tutti gli output principali prodotti dal builder, così da poterli riutilizzare nel notebook ML o in analisi successive.

In [10]:
# ── Assembla panel_macro da macro_df ─────────────────────────────────────────
if 'macro_df' in globals():
    panel_macro = macro_df.copy().reset_index()
    panel_macro.rename(columns={'index':'Date'}, inplace=True)
    panel_macro['Date'] = pd.to_datetime(panel_macro['Date'])
    # Aggiungi DJIA_Return mensile
    if 'DJIA_Real' in panel_macro.columns:
        panel_macro['DJIA_Return'] = panel_macro['DJIA_Real'].pct_change()
else:
    panel_macro = pd.DataFrame()

print(f"✓ panel_macro: {panel_macro.shape}")
print()

# ── Salva i tre DataFrame ────────────────────────────────────────────────────
panel_returns.to_csv('panel_returns.csv', index=False)
panel_valuation.to_csv('panel_valuation.csv', index=False)
if not panel_macro.empty:
    panel_macro.to_csv('panel_macro.csv', index=False)
price_df.to_csv('price_matrix_all49.csv')
download_metadata.to_csv('download_metadata.csv', index=False)

print("✓ panel_returns.csv     →", panel_returns.shape)
print("  Colonne:", list(panel_returns.columns))
print()
print("✓ panel_valuation.csv   →", panel_valuation.shape)
print("  Colonne:", list(panel_valuation.columns))
print()
if not panel_macro.empty:
    print("✓ panel_macro.csv       →", panel_macro.shape)
    print("  Colonne:", list(panel_macro.columns))
print()
print("✓ price_matrix_all49.csv →", price_df.shape)
print()

try:
    from google.colab import files
    for fn in ['panel_returns.csv','panel_valuation.csv','panel_macro.csv',
               'price_matrix_all49.csv','download_metadata.csv']:
        try: files.download(fn)
        except: pass
    print("✓ Download avviato (Colab)")
except ImportError:
    print("(Non su Colab: file salvati nella directory corrente)")


✓ Salvato: panel_dataset_all49_2005_2026.csv
✓ Salvato: price_matrix_all49.csv
✓ Salvato: download_metadata_all49_2005_2026.csv
  Shape panel: (12544, 19)

Colonne del panel:
  Date                      datetime64[ns]  null: 0
  Ticker                    object        null: 0
  In_Index                  int64         null: 0
  Price                     float64       null: 701
  Return_monthly            float64       null: 750
  Price_Available           int64         null: 0
  Include_In_Training       int64         null: 0
  Missing_In_Index          int64         null: 0
  Missing_Reason            object        null: 0
  Weight_PW                 float64       null: 4,917
  Weight_PW_Clean           float64       null: 4,917
  Weight_EW                 float64       null: 4,864
  Weight_EW_Clean           float64       null: 4,917
  RF_Monthly                float64       null: 0
  Sum_Prices_InIndex        float64       null: 0
  Sum_Prices_Eligible       float64       null: 0
  E

---
## 9. Diagnostica e Verifica Qualità

Controlli rapidi per verificare la correttezza del dataset prima di usarlo nell'analisi ML, con focus specifico su:

- copertura dei ticker in indice
- patch effettiva di **WBA**
- esclusione esplicita di **GM** dal sotto-universo clean quando il prezzo manca
- coerenza dei pesi `Weight_PW_Clean` e `Weight_EW_Clean`
- conferma operativa che il benchmark corretto per il modello ML è **equal-weight clean**


## 11. Diagnostica del dataset

Qui eseguiamo controlli di qualità sul dataset finale:
- numero di titoli in indice per mese
- copertura di WBA
- gestione di GM
- coerenza dei pesi clean
- eventuali mesi problematici

## 11b. Qualità della Ricostruzione dell'Indice DJIA

Confrontiamo la nostra ricostruzione con il DJIA reale (FRED).

> ⚠️ **Nota critica**: la ricostruzione soffre di un bias positivo significativo nel periodo post-2015,
> causato principalmente da **WBA** (Walgreens Boots Alliance, 2018-2024, -97% circa) 
> e **DWDP/DOW** che mancano di prezzi scaricabili. Queste azioni con performance 
> negative vengono trattate come 0-return invece di essere ponderate correttamente.
> Il DJIA reale (da FRED) è il benchmark corretto da usare nel modello ML.


In [ ]:
# ── Qualità ricostruzione DJIA con pesi lagged ───────────────────────────────
#
# R_DJIA_proxy(t) = Σ_i [ Weight_PW_begin_i(t) × Return_i(t) ]
# dove Weight_PW_begin è il peso del mese PRECEDENTE (t-1) — come fa il vero DJIA.

if 'macro_df' in globals() and 'DJIA_Real' in macro_df.columns:
    djia_real     = macro_df['DJIA_Real'].dropna().sort_index()
    djia_real_ret = djia_real.pct_change().dropna()
    djia_real_ret.index = pd.to_datetime(djia_real_ret.index) + pd.offsets.MonthEnd(0)

    in_idx = panel_returns[panel_returns['In_Index']==1].copy()

    # Ricostruzione con pesi lagged (Weight_PW_begin) — CORRETTA
    rec_begin = (in_idx.dropna(subset=['Weight_PW_begin','Return_monthly'])
                  .groupby('Date')
                  .apply(lambda g: (g['Weight_PW_begin'] * g['Return_monthly']).sum())
                  .rename('Rec_begin'))

    # Ricostruzione con pesi correnti (Weight_PW_end) — per confronto
    rec_end   = (in_idx.dropna(subset=['Weight_PW_end','Return_monthly'])
                  .groupby('Date')
                  .apply(lambda g: (g['Weight_PW_end'] * g['Return_monthly']).sum())
                  .rename('Rec_end'))

    common = rec_begin.index.intersection(djia_real_ret.index)
    df_cmp = pd.DataFrame({
        'Rec_Lagged':  rec_begin.reindex(common),
        'Rec_Current': rec_end.reindex(common),
        'Real':        djia_real_ret.reindex(common),
    }).dropna()

    corr_lag = df_cmp['Rec_Lagged'].corr(df_cmp['Real'])
    corr_cur = df_cmp['Rec_Current'].corr(df_cmp['Real'])
    te_lag   = (df_cmp['Rec_Lagged']  - df_cmp['Real']).std() * np.sqrt(12) * 100
    te_cur   = (df_cmp['Rec_Current'] - df_cmp['Real']).std() * np.sqrt(12) * 100

    print("=" * 65)
    print("QUALITÀ RICOSTRUZIONE DJIA")
    print("=" * 65)
    print(f"{'':25} {'Pesi lagged (begin)':>20} {'Pesi correnti (end)':>20}")
    print("-" * 65)
    print(f"{'Correlazione':25} {corr_lag:>20.4f} {corr_cur:>20.4f}")
    print(f"{'Tracking Error annuo (%)':25} {te_lag:>20.2f} {te_cur:>20.2f}")
    print("-" * 65)

    # Cumulativo
    cum_lag = (1 + df_cmp['Rec_Lagged']).cumprod()
    cum_cur = (1 + df_cmp['Rec_Current']).cumprod()
    cum_rea = (1 + df_cmp['Real']).cumprod()
    print(f"{'Rendimento cumulativo:':25}")
    print(f"  DJIA Reale       : {(cum_rea.iloc[-1]-1)*100:+.1f}%")
    print(f"  Proxy lagged     : {(cum_lag.iloc[-1]-1)*100:+.1f}%")
    print(f"  Proxy corrente   : {(cum_cur.iloc[-1]-1)*100:+.1f}%")
    print()

    # Dettaglio gap per anno
    df_cmp['Year'] = pd.to_datetime(df_cmp.index).year
    annual = df_cmp.groupby('Year').apply(
        lambda g: pd.Series({
            'Corr': g['Rec_Lagged'].corr(g['Real']),
            'TE%':  (g['Rec_Lagged']-g['Real']).std()*np.sqrt(12)*100,
            'Gap%': ((1+g['Rec_Lagged']).prod()-(1+g['Real']).prod())*100,
            'N':    len(g),
        })
    )
    print("Dettaglio per anno (pesi lagged):")
    print(f"  {'Anno':5} {'Corr':>6} {'TE%':>7} {'Gap%':>8} {'N':>4}")
    print("  " + "-"*34)
    for yr, row in annual.iterrows():
        flag = ' ⚠' if abs(row['Gap%']) > 5 else ''
        print(f"  {yr:5} {row['Corr']:>6.3f} {row['TE%']:>7.2f} {row['Gap%']:>8.2f}%{flag}")

    print()
    print("⚠  Gap anni 2005-2010: dovuto a GM mancante (old GM bankrupt 2009)")
    print("   → In quel periodo il proxy sovrastima perché non cattura -97% GM")

else:
    print("⚠ macro_df non disponibile — esegui la cella FRED")


In [11]:

print("=" * 70)
print("DIAGNOSTICA DATASET")
print("=" * 70)

# ── 1. Titoli membri del DJIA per mese ───────────────────────────────────────
members_per_month = panel[panel['In_Index'] == 1].groupby('Date')['Ticker'].count()
print("\n[1] Titoli membri del DJIA per mese:")
print(f"    Min: {members_per_month.min()} | Max: {members_per_month.max()} | Atteso: 27–30")
print(f"    Mesi con ≠ 30 titoli: {(members_per_month != 30).sum()}")

# ── 2. Titoli eligibili nel sotto-universo clean ─────────────────────────────
eligible_per_month = (
    panel[panel['Include_In_Training'] == 1]
    .groupby('Date')['Ticker']
    .count()
)
print("\n[2] Titoli eligibili nel sotto-universo clean:")
print(f"    Min: {eligible_per_month.min()} | Max: {eligible_per_month.max()}")
print(f"    Mesi con meno di 30 eligibili: {(eligible_per_month < 30).sum()}")

# ── 3. Missing while in index ────────────────────────────────────────────────
missing_in_idx = (
    panel[panel['Missing_In_Index'] == 1]
    .groupby('Ticker')
    .size()
    .sort_values(ascending=False)
)
print("\n[3] Ticker in indice ma senza prezzo:")
if len(missing_in_idx) == 0:
    print("    ✓ Nessun caso")
else:
    print(missing_in_idx.to_string())

# ── 4. Focus GM ──────────────────────────────────────────────────────────────
gm_missing = panel[(panel['Ticker'] == 'GM') & (panel['Missing_In_Index'] == 1)]
print("\n[4] Focus GM:")
print(f"    Mesi GM in indice senza prezzo: {len(gm_missing)}")
if len(gm_missing) > 0:
    print(f"    Primo mese: {gm_missing['Date'].min().date()} | Ultimo mese: {gm_missing['Date'].max().date()}")
    print("    Trattamento: escluso dal sotto-universo clean / training")

# ── 5. Focus WBA ─────────────────────────────────────────────────────────────
print("\n[5] Focus WBA:")
if 'WBA' in price_df.columns:
    wba_non_null = int(price_df['WBA'].notna().sum())
    print(f"    Mesi con prezzo disponibile in price_df: {wba_non_null}")
    meta_wba = download_metadata.loc[download_metadata['Ticker'] == 'WBA']
    if not meta_wba.empty:
        print(f"    Fonte finale: {meta_wba['Source'].iloc[0]}")
        print(f"    First_Date: {meta_wba['First_Date'].iloc[0]} | Last_Date: {meta_wba['Last_Date'].iloc[0]}")
    if wba_non_null == 0:
        print("    ⚠ WBA ancora mancante: verificare la presenza del file locale pulito o la fonte alternativa")
else:
    print("    ⚠ Colonna WBA non presente in price_df")

# ── 6. Verifica somma pesi clean ─────────────────────────────────────────────
pw_clean_sum = panel[panel['Include_In_Training'] == 1].groupby('Date')['Weight_PW_Clean'].sum()
ew_clean_sum = panel[panel['Include_In_Training'] == 1].groupby('Date')['Weight_EW_Clean'].sum()
print("\n[6] Somma pesi clean per mese:")
print(f"    Weight_PW_Clean  -> min {pw_clean_sum.min():.6f} | max {pw_clean_sum.max():.6f}")
print(f"    Weight_EW_Clean  -> min {ew_clean_sum.min():.6f} | max {ew_clean_sum.max():.6f}")

# ── 7. Sharpe coverage ───────────────────────────────────────────────────────
sharpe_cover = panel['Sharpe_12m'].notna().mean()
print("\n[7] Copertura Sharpe_12m:")
print(f"    {sharpe_cover:.1%} delle righe")

# ── 8. Benchmark recommendation ──────────────────────────────────────────────
print("\n[8] Benchmark raccomandato per ML:")
print("    ✓ Usare Weight_EW_Clean come benchmark operativo")
print("    ℹ Weight_PW_Clean resta utile per analisi diagnostica / sensibilità")
print("    ℹ Il DJI ufficiale non è replicato dal simple price-weighted a causa del Dow Divisor")

# ── 9. Prime anomalie documentate ────────────────────────────────────────────
anomalies = panel[
    (panel['Missing_In_Index'] == 1) |
    ((panel['In_Index'] == 1) & (panel['Include_In_Training'] == 0))
][['Date','Ticker','In_Index','Price','Missing_Reason','Include_In_Training']].head(20)

print("\n[9] Prime righe anomalie / esclusioni clean:")
if anomalies.empty:
    print("    ✓ Nessuna anomalia")
else:
    print(anomalies.to_string(index=False))

print()
print("=" * 70)
print("✓ Diagnostica completata.")


DIAGNOSTICA DATASET

[1] Titoli membri del DJIA per mese:
    Min: 30 | Max: 30 | Atteso: 27–30
    Mesi con ≠ 30 titoli: 0

[2] Titoli eligibili nel sotto-universo clean:
    Min: 29 | Max: 30
    Mesi con meno di 30 eligibili: 53

[3] Ticker in indice ma senza prezzo:
Ticker
GM    53

[4] Focus GM:
    Mesi GM in indice senza prezzo: 53
    Primo mese: 2005-01-31 | Ultimo mese: 2009-05-31
    Trattamento: escluso dal sotto-universo clean / training

[5] Focus WBA:
    Mesi con prezzo disponibile in price_df: 69
    Fonte finale: Embedded investing backup
    First_Date: 2018-06-30 00:00:00 | Last_Date: 2024-02-29 00:00:00

[6] Somma pesi clean per mese:
    Weight_PW_Clean  -> min 0.999995 | max 1.000004
    Weight_EW_Clean  -> min 0.999990 | max 1.000007

[7] Copertura Sharpe_12m:
    89.7% delle righe

[8] Benchmark raccomandato per ML:
    ✓ Usare Weight_EW_Clean come benchmark operativo
    ℹ Weight_PW_Clean resta utile per analisi diagnostica / sensibilità
    ℹ Il DJI ufficiale

---
## 12. Visualizzazioni

Grafici esplorativi su **prezzi**, **rendimenti** e **valutazioni** del dataset DJIA.
Ogni chart è salvato come PNG nella directory corrente e mostrato inline.


In [ ]:
import matplotlib
matplotlib.use('Agg')   # backend non-interattivo (compatibile Jupyter + headless)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
})
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

print("✓ matplotlib configurato")

# ── Prepara serie utili ───────────────────────────────────────────────────────
panel_dt = panel.copy()
panel_dt['Date'] = pd.to_datetime(panel_dt['Date'])

# DJIA reale mensile
djia_real_plot = macro_df['DJIA_Real'].dropna().sort_index() if 'macro_df' in globals() else pd.Series(dtype=float)

# DJIA proxy mensile (da panel: primo valore non-NaN per data)
if 'DJIA_Proxy' in panel_dt.columns:
    djia_proxy_plot = (panel_dt.groupby('Date')['DJIA_Proxy'].first()
                       .dropna().sort_index())
else:
    djia_proxy_plot = pd.Series(dtype=float)

# Rendimenti mensili in-index
ret_panel = panel_dt[(panel_dt['In_Index']==1) & panel_dt['Return_monthly'].notna()].copy()
ret_panel['Year'] = ret_panel['Date'].dt.year

# Valutazioni mensili mediane in-index
val_monthly = (panel_dt[panel_dt['In_Index']==1]
               .groupby('Date')[['PE_Ratio','PB_Ratio','EV_EBITDA','Net_Margin']]
               .median())

print(f"  DJIA reale: {len(djia_real_plot)} mesi")
print(f"  DJIA proxy: {len(djia_proxy_plot)} mesi")
print(f"  Righe rendimenti in-index: {len(ret_panel):,}")
print(f"  Mesi con valutazioni: {val_monthly.dropna(how='all').shape[0]}")


In [ ]:
# ── CHART 1: DJIA Reale vs Proxy (normalizzati a 100) ───────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          gridspec_kw={'height_ratios': [3, 1]})

ax1, ax2 = axes

# Normalizza a 100 al primo dato comune
common_idx = djia_real_plot.index.intersection(djia_proxy_plot.index)
if len(common_idx) > 5:
    t0 = common_idx[0]
    real_n  = djia_real_plot.loc[common_idx]  / djia_real_plot.loc[t0] * 100
    proxy_n = djia_proxy_plot.loc[common_idx] / djia_proxy_plot.loc[t0] * 100

    ax1.plot(real_n.index,  real_n.values,  lw=1.8, color='#1f77b4', label='DJIA Reale')
    ax1.plot(proxy_n.index, proxy_n.values, lw=1.2, color='#ff7f0e',
             alpha=0.85, linestyle='--', label='DJIA Proxy (chain-linked)')
    ax1.set_title('DJIA Reale vs Proxy — normalizzati a 100 al primo dato comune')
    ax1.set_ylabel('Livello (base 100)')
    ax1.legend(frameon=False)

    # Differenza %
    diff = proxy_n - real_n
    ax2.fill_between(diff.index, diff.values, 0,
                     where=diff > 0, color='#ff7f0e', alpha=0.4, label='Proxy > Reale')
    ax2.fill_between(diff.index, diff.values, 0,
                     where=diff < 0, color='#1f77b4', alpha=0.4, label='Proxy < Reale')
    ax2.axhline(0, color='k', lw=0.6)
    ax2.set_ylabel('Differenza (punti)')
    ax2.set_xlabel('Data')
    ax2.legend(frameon=False, fontsize=8)

    # Evidenzia zona gap GM
    gm_start = pd.Timestamp('2005-01-01')
    gm_end   = pd.Timestamp('2010-11-01')
    for ax in axes:
        ax.axvspan(gm_start, gm_end, color='red', alpha=0.06, zorder=0)
    ax1.annotate('Gap GM\n(2005-2010)', xy=(pd.Timestamp('2007-06-01'), ax1.get_ylim()[0]),
                 fontsize=7, color='red', ha='center')
else:
    ax1.text(0.5, 0.5, 'DJIA reale non disponibile\n(manca FRED o Excel)',
             ha='center', va='center', transform=ax1.transAxes)

plt.tight_layout()
plt.savefig('chart1_djia_proxy_vs_real.png', bbox_inches='tight')
plt.show()
print("✓ chart1_djia_proxy_vs_real.png salvato")


In [ ]:
# ── CHART 2: Rendimenti Mensili — Boxplot per Anno ──────────────────────────
years = sorted(ret_panel['Year'].unique())
ret_by_year = [ret_panel[ret_panel['Year']==y]['Return_monthly'].values * 100
               for y in years]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(ret_by_year, labels=years, patch_artist=True,
                medianprops=dict(color='black', lw=1.5),
                whiskerprops=dict(lw=0.8), capprops=dict(lw=0.8),
                flierprops=dict(marker='.', markersize=2, alpha=0.4))

# Colora box: verde se anno positivo (mediana>0), rosso altrimenti
for patch, data in zip(bp['boxes'], ret_by_year):
    med = float(pd.Series(data).median())
    patch.set_facecolor('#c8e6c9' if med >= 0 else '#ffcdd2')
    patch.set_alpha(0.8)

ax.axhline(0, color='k', lw=0.8, linestyle='--')
ax.set_xlabel('Anno')
ax.set_ylabel('Rendimento mensile (%)')
ax.set_title('Distribuzione Rendimenti Mensili per Anno — componenti DJIA in-index')

# Aggiungi linea rendimento DJIA reale per confronto
if not djia_real_plot.empty:
    djia_ret_by_year = (djia_real_plot.pct_change().dropna() * 100)
    djia_ret_by_year.index = pd.to_datetime(djia_ret_by_year.index)
    yr_means = djia_ret_by_year.groupby(djia_ret_by_year.index.year).mean()
    yr_pos = {y: i+1 for i, y in enumerate(years)}
    xs = [yr_pos[y] for y in yr_means.index if y in yr_pos]
    ys = [yr_means[y] for y in yr_means.index if y in yr_pos]
    ax.plot(xs, ys, 'o-', color='#1f77b4', lw=1.5, ms=4,
            label='Rendimento medio mensile DJIA reale', zorder=5)
    ax.legend(frameon=False, fontsize=8)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart2_returns_boxplot.png', bbox_inches='tight')
plt.show()
print("✓ chart2_returns_boxplot.png salvato")


In [ ]:
# ── CHART 3: Valutazioni Mediane DJIA nel Tempo ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.flatten()

metrics = [
    ('PE_Ratio',  'P/E Ratio',         '#1f77b4', (0, 60)),
    ('PB_Ratio',  'P/B Ratio',         '#ff7f0e', (0, 10)),
    ('EV_EBITDA', 'EV / EBITDA',       '#2ca02c', (0, 30)),
    ('Net_Margin','Net Margin (mediana)','#d62728', (-0.1, 0.5)),
]

for ax, (col, title, color, ylim) in zip(axes, metrics):
    if col not in val_monthly.columns:
        ax.text(0.5, 0.5, f'{col}\nnon disponibile', ha='center', va='center',
                transform=ax.transAxes)
        continue
    s = val_monthly[col].dropna()
    if s.empty:
        ax.text(0.5, 0.5, f'{col}\nnessun dato', ha='center', va='center',
                transform=ax.transAxes)
        continue

    ax.plot(s.index, s.values, color=color, lw=1.5)

    # Banda ±1 std rolling 12 mesi
    roll_mean = s.rolling(12, min_periods=6).mean()
    roll_std  = s.rolling(12, min_periods=6).std()
    ax.fill_between(s.index,
                    (roll_mean - roll_std).clip(lower=ylim[0]),
                    (roll_mean + roll_std).clip(upper=ylim[1]),
                    alpha=0.2, color=color)

    # Evidenzia recessioni NBER
    if 'macro_df' in globals() and 'Recession' in macro_df.columns:
        rec = macro_df['Recession'].reindex(s.index).fillna(0)
        ax.fill_between(s.index, ylim[0], ylim[1],
                        where=rec > 0.5, alpha=0.1, color='gray', label='Recessione NBER')

    ax.set_title(title)
    ax.set_ylim(ylim)
    ax.set_ylabel(title)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Valutazioni Mediane dei Componenti DJIA (In-Index)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('chart3_valuations_over_time.png', bbox_inches='tight')
plt.show()
print("✓ chart3_valuations_over_time.png salvato")


In [ ]:
# ── CHART 4: Heatmap Rendimenti Annuali per Ticker (ultimi 10 anni) ──────────
import numpy as np

# Seleziona ticker con almeno 60 mesi di rendimenti in-index
recent_tickers = (panel_dt[(panel_dt['In_Index']==1) &
                            (panel_dt['Date'] >= pd.Timestamp('2015-01-01'))]
                  .groupby('Ticker')['Return_monthly']
                  .apply(lambda x: x.notna().sum()))
active_tickers = recent_tickers[recent_tickers >= 36].index.tolist()

# Calcola rendimento annualizzato per ticker x anno
ann_ret = (panel_dt[(panel_dt['In_Index']==1) &
                     panel_dt['Ticker'].isin(active_tickers)]
           .assign(Year=lambda d: d['Date'].dt.year)
           .groupby(['Ticker','Year'])['Return_monthly']
           .apply(lambda x: (1 + x.dropna()).prod() - 1 if len(x.dropna()) >= 6 else np.nan)
           .unstack('Year')
           .dropna(how='all'))

years_plot = sorted([y for y in ann_ret.columns if 2014 <= y <= 2025])
heat_data  = ann_ret[years_plot].astype(float)
heat_data  = heat_data.dropna(how='all').sort_index()

if not heat_data.empty:
    fig, ax = plt.subplots(figsize=(14, max(5, len(heat_data)*0.35)))

    vmax = min(abs(heat_data.values[~np.isnan(heat_data.values)]).max(), 0.80)
    im   = ax.imshow(heat_data.values, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')

    ax.set_xticks(range(len(years_plot)))
    ax.set_xticklabels(years_plot, fontsize=8)
    ax.set_yticks(range(len(heat_data.index)))
    ax.set_yticklabels(heat_data.index, fontsize=8)

    # Valori nelle celle
    for i in range(len(heat_data.index)):
        for j in range(len(years_plot)):
            v = heat_data.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v*100:.0f}%', ha='center', va='center',
                        fontsize=6.5, color='black' if abs(v) < 0.4 else 'white')

    plt.colorbar(im, ax=ax, label='Rendimento annuale', fraction=0.02, pad=0.02,
                 format=lambda x, _: f'{x*100:.0f}%')
    ax.set_title('Rendimenti Annuali per Componente DJIA (In-Index, 2015-2025)', fontsize=10)
    plt.tight_layout()
    plt.savefig('chart4_annual_returns_heatmap.png', bbox_inches='tight')
    plt.show()
    print("✓ chart4_annual_returns_heatmap.png salvato")
else:
    print("  Dati insufficienti per la heatmap")
